# Q5 — 멀티태스크: 결함 종류 + 공정 원인 추론
원래 기획의 Phase 2 목표. 하나의 모델(백본 공유)이 두 가지를 동시에 예측합니다.
1) **결함 종류** 9-class, 2) **공정 원인** 카테고리. 추론 시 결함→원인→**조치 가이드**까지 출력.
원인 라벨은 `configs/mappings/defect_to_cause.yaml`(제안서 2.1절 표 기반)에서 생성합니다.
전제: `data/processed/*.npz` 존재. 위에서부터 셀 실행.

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path
_p = Path.cwd()
for cand in [_p, _p.parent, _p.parent.parent]:
    if (cand/"utils").is_dir():
        sys.path.insert(0, str(cand)); PROJ=cand; break
else:
    PROJ=_p
from utils.korean_font import set_korean_font
set_korean_font()

import numpy as np, matplotlib.pyplot as plt, time, yaml
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import f1_score, accuracy_score, classification_report

CLASSES=["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM=len(CLASSES)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
PROC=PROJ/"data"/"processed"; CKPT=PROJ/"models"/"checkpoints"; CKPT.mkdir(parents=True,exist_ok=True)
FIG=PROJ/"results"/"figures"; FIG.mkdir(parents=True,exist_ok=True)
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- 1. 결함 -> 공정 원인 매핑 로드 ---
mp = yaml.safe_load(open(PROJ/"configs"/"mappings"/"defect_to_cause.yaml", encoding="utf-8"))
CAUSE_NAMES = list(mp["causes"]) + ["normal"]   # 8개 원인 + 정상 = 9
cause2idx = {c:i for i,c in enumerate(CAUSE_NAMES)}
NUM_CAUSE = len(CAUSE_NAMES)

def defect_to_cause_idx(defect_name):
    if defect_name == "none":
        return cause2idx["normal"]
    return cause2idx[mp["pattern_to_cause"][defect_name]["primary"]]

# 결함 인덱스 -> 원인 인덱스 변환표
cause_of = np.array([defect_to_cause_idx(CLASSES[d]) for d in range(NUM)], dtype=np.int64)
print("결함 -> 원인 매핑")
for d in range(NUM):
    print(f"  {CLASSES[d]:10s} -> {CAUSE_NAMES[cause_of[d]]}")
ACTION = mp["action_guide"]

In [ ]:
# --- 2. 데이터 로드 + 원인 라벨 생성 ---
def load_split(n):
    d=np.load(PROC/f"wafer_{n}.npz", allow_pickle=True); return d["X"], d["y"]
Xtr,ytr=load_split("train"); Xva,yva=load_split("val"); Xte,yte=load_split("test")
# 결함 라벨에서 원인 라벨 파생
ctr, cva, cte = cause_of[ytr], cause_of[yva], cause_of[yte]
counts=np.bincount(ytr, minlength=NUM)
print("train:", Xtr.shape, "| defect classes:", NUM, "| cause classes:", NUM_CAUSE)

BATCH=256; EPOCHS=6; LR=3e-4; ALPHA=0.5  # ALPHA = 원인 손실 비중
FOCAL_GAMMA=2.0; CB_BETA=0.999
torch.manual_seed(42); np.random.seed(42)

class WaferDS(Dataset):
    def __init__(s, X, yd, yc, train=False): s.X=X; s.yd=yd; s.yc=yc; s.train=train
    def __len__(s): return len(s.X)
    def __getitem__(s, i):
        img=s.X[i].astype(np.float32)/2.0
        if s.train:
            k=np.random.randint(0,4)
            if k: img=np.rot90(img,k).copy()
            if np.random.rand()<0.5: img=np.fliplr(img).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy()
        img=(img-0.5)/0.5
        img=np.expand_dims(img,0).repeat(3,0)
        return torch.from_numpy(img), int(s.yd[i]), int(s.yc[i])

mk=lambda X,yd,yc,tr: DataLoader(WaferDS(X,yd,yc,tr), batch_size=BATCH, shuffle=tr, num_workers=0, pin_memory=(DEVICE=="cuda"))
tr_loader=mk(Xtr,ytr,ctr,True); va_loader=mk(Xva,yva,cva,False); te_loader=mk(Xte,yte,cte,False)

## 3. 멀티태스크 모델
ResNet18 백본을 공유하고, 그 위에 두 개의 분류 헤드를 둡니다. 백본이 추출한 공통 특징을 두 태스크가 함께 사용하므로 효율적이고, 결함 정보가 원인 추정을 돕습니다.

In [ ]:
# --- 3. 백본 공유 + 2개 헤드 ---
class MultiTaskNet(nn.Module):
    def __init__(s, n_defect, n_cause):
        super().__init__()
        try: bb=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception: bb=models.resnet18(pretrained=True)
        in_feat=bb.fc.in_features
        bb.fc=nn.Identity()           # 백본은 특징 추출기로만
        s.backbone=bb
        s.defect_head=nn.Linear(in_feat, n_defect)
        s.cause_head =nn.Linear(in_feat, n_cause)
    def forward(s, x):
        f=s.backbone(x)
        return s.defect_head(f), s.cause_head(f)

net=MultiTaskNet(NUM, NUM_CAUSE).to(DEVICE)

# 결함 헤드용 class-balanced 가중치 (Q3와 동일 전략)
eff=1.0-np.power(CB_BETA,counts); cbw=(1-CB_BETA)/np.maximum(eff,1e-12)
cbw=(cbw/cbw.sum()*NUM).astype(np.float32)
defect_w=torch.tensor(cbw, device=DEVICE)

class FocalLoss(nn.Module):
    def __init__(s, weight=None, gamma=2.0): super().__init__(); s.w=weight; s.g=gamma
    def forward(s, logits, target):
        logp=F.log_softmax(logits,1); ce=F.nll_loss(logp,target,weight=s.w,reduction="none")
        pt=logp.exp().gather(1,target.unsqueeze(1)).squeeze(1)
        return ((1-pt)**s.g*ce).mean()

defect_crit=FocalLoss(defect_w, FOCAL_GAMMA)
cause_crit =nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print("멀티태스크 모델 준비 (defect head + cause head, 백본 공유)")

In [ ]:
# --- 4. 학습 (total = focal_defect + ALPHA * ce_cause) ---
@torch.no_grad()
def evaluate(loader):
    net.eval(); yd=[];yc=[];pd=[];pc=[]
    for xb,ydb,ycb in loader:
        od,oc=net(xb.to(DEVICE))
        pd.append(od.argmax(1).cpu().numpy()); pc.append(oc.argmax(1).cpu().numpy())
        yd.append(ydb.numpy()); yc.append(ycb.numpy())
    yd=np.concatenate(yd);yc=np.concatenate(yc);pd=np.concatenate(pd);pc=np.concatenate(pc)
    return (f1_score(yd,pd,average="macro"), f1_score(yc,pc,average="macro"),
            accuracy_score(yd,pd), accuracy_score(yc,pc), yd,yc,pd,pc)

best=-1; best_path=CKPT/"q5_multitask_resnet18.pt"
for ep in range(1,EPOCHS+1):
    net.train(); t0=time.time(); run=0.0
    for bi,(xb,ydb,ycb) in enumerate(tr_loader):
        xb,ydb,ycb=xb.to(DEVICE),ydb.to(DEVICE),ycb.to(DEVICE)
        od,oc=net(xb)
        loss=defect_crit(od,ydb)+ALPHA*cause_crit(oc,ycb)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); run+=loss.item()
        if bi%100==0: print(f"  ep{ep} {bi}/{len(tr_loader)} loss={loss.item():.3f}", end="\r")
    scheduler.step()
    df1,cf1,dacc,cacc,*_=evaluate(va_loader)
    flag=""
    if df1>best: best=df1; torch.save({"model":net.state_dict(),"classes":CLASSES,"causes":CAUSE_NAMES}, best_path); flag="  <- best"
    print(f"[ep{ep}] val defectF1={df1:.3f} causeF1={cf1:.3f} | defAcc={dacc:.3f} causeAcc={cacc:.3f} | {time.time()-t0:.0f}s{flag}")
print("\nbest val defect macro-F1:", round(best,4))

In [ ]:
# --- 5. test 평가 (두 헤드 모두) ---
net.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
df1,cf1,dacc,cacc,yd,yc,pd,pc=evaluate(te_loader)
print(f"[결함 종류]  macro-F1={df1:.3f} | accuracy={dacc:.3f}")
print(f"[공정 원인]  macro-F1={cf1:.3f} | accuracy={cacc:.3f}\n")
print("=== 공정 원인 분류 리포트 ===")
present=sorted(set(yc.tolist())|set(pc.tolist()))
print(classification_report(yc,pc,labels=present,target_names=[CAUSE_NAMES[i] for i in present],digits=3,zero_division=0))

In [ ]:
# --- 6. 추론 데모: 웨이퍼 -> 결함 + 원인 + 조치 가이드 ---
@torch.no_grad()
def diagnose(wafer_u8):
    img=(wafer_u8.astype(np.float32)/2.0-0.5)/0.5
    x=torch.from_numpy(np.expand_dims(img,0).repeat(3,0)).unsqueeze(0).to(DEVICE)
    od,oc=net(x)
    di=int(od.argmax(1)); ci=int(oc.argmax(1))
    dp=float(od.softmax(1)[0,di]); cp=float(oc.softmax(1)[0,ci])
    cname=CAUSE_NAMES[ci]
    return {"defect":CLASSES[di],"defect_p":dp,"cause":cname,"cause_p":cp,
            "action":ACTION.get(cname,"-")}

# 결함 샘플 3개 진단
rng=np.random.default_rng(7)
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,idx in zip(axes, rng.choice(np.where(yte!=8)[0], 3, replace=False)):
    r=diagnose(Xte[idx])
    ax.imshow(Xte[idx], cmap="viridis", vmin=0, vmax=2); ax.axis("off")
    ax.set_title(f"실제:{CLASSES[yte[idx]]}", fontsize=10)
    print(f"[실제 {CLASSES[yte[idx]]:9s}] 결함예측={r['defect']:9s}({r['defect_p']*100:.0f}%) "
          f"원인={r['cause']:16s}({r['cause_p']*100:.0f}%)")
    print(f"     조치: {r['action']}\n")
plt.suptitle("결함 + 공정 원인 동시 진단 데모", fontsize=13); plt.tight_layout(); plt.show()

## 정리

- 하나의 모델이 **결함 종류**와 **공정 원인**을 동시에 예측하고, 원인에 맞는 **조치 가이드**까지 제시합니다. 원래 기획(제안서 2.1절)의 멀티태스크 목표를 구현한 것입니다.
- 본 데이터셋에서는 각 결함이 하나의 주요 원인에 대응(1:1)하므로, 원인 헤드 성능은 결함 헤드와 유사하게 나옵니다. 원인 헤드의 가치는 (1) 결과를 곧바로 조치로 연결하는 실용성, (2) 혼합 결함(예: MixedWM38)처럼 원인이 1:다로 늘어날 때 그대로 확장되는 구조에 있습니다.
- 다음 확장: 혼합 결함 데이터셋 추가, 원인 다중 라벨(sigmoid+BCE)화, 조치 가이드를 포함한 진단 리포트 자동 생성.